# NB04 — Ablation Studies

**GraphSentry: A Unified GNN Framework for Blockchain Illicit Activity Detection**

Systematically isolates the contribution of individual GraphSentry components by
disabling one at a time while holding all other variables constant.

**Ablation groups:**
- **A: Feature sets** — Full (44-dim) vs Anonymous-only (43-dim) vs Degree-only (1-dim)
- **B: Pooling strategy** — Global Mean vs Global Max vs Global Add
- **C: Architecture depth** — 2 vs 3 vs 4 GCN layers

Control = Model A from NB02 (full features, mean pool, 3 layers).
Sampling ablation is captured cross-notebook: NB02 (GraphSAINT) vs NB03 (DataLoader).

**Requires:** Artefacts from `NB01_data_pipeline.ipynb`

**Produces:**
- `ablation_results.json` — metrics for all ablation experiments

## 1. Configuration

In [36]:
MAX_EPOCHS = 60
PATIENCE = 10
BATCH_SIZE = 64
LR = 0.005
LR_FACTOR = 0.5
LR_PATIENCE = 5
HIDDEN_DIM = 128
DROPOUT = 0.5
SEED = 42

BASE_PATH = '/content/drive/MyDrive/GraphSentry'
PROCESSED_PATH = f'{BASE_PATH}/data/processed'

## 2. Environment + load artefacts

In [37]:
!pip install -q uv
!uv pip install --system torch torch-geometric numpy pandas tqdm scikit-learn

from google.colab import drive
import os, json, time, copy

import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.utils import degree
from torch_geometric.loader import DataLoader
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix, average_precision_score
)

np.random.seed(SEED)
torch.manual_seed(SEED)

drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

bg = torch.load(os.path.join(PROCESSED_PATH, 'background_graph.pt'), weights_only=False)
meta = torch.load(os.path.join(PROCESSED_PATH, 'cc_metadata.pt'), weights_only=False)
feat_anon = torch.load(os.path.join(PROCESSED_PATH, 'features_anonymous.pt'), weights_only=False)

with open(os.path.join(PROCESSED_PATH, 'dataset_stats.json'), 'r') as f:
    stats = json.load(f)

ANON_DIM = stats['feature_dims']['anonymous']
print(f"Dataset: {stats['total_ccs']} CCs, {stats['total_nodes']} nodes, anon dim = {ANON_DIM}")

Using Python 3.12.13 environment at: /usr
Checked 6 packages in 90ms
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Dataset: 5500 CCs, 39819 nodes, anon dim = 43


## 3. Build datasets by feature mode

In [38]:
POOL_FN = {
    'mean': global_mean_pool,
    'max':  global_max_pool,
    'add':  global_add_pool,
}


def build_pyg_graph(cc_id, feature_mode='full'):
    cc_nodes = sorted(bg['cc_to_nodes'][cc_id])
    local_map = {g: l for l, g in enumerate(cc_nodes)}
    adj = bg['adj_list']

    x_anon = feat_anon[cc_nodes].clone()

    node_set = set(cc_nodes)
    local_src, local_dst = [], []
    for g_src in cc_nodes:
        for g_dst in adj[g_src]:
            if g_dst in node_set and g_dst in local_map:
                local_src.append(local_map[g_src])
                local_dst.append(local_map[g_dst])

    if len(local_src) == 0:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        edge_index = torch.tensor([local_src, local_dst], dtype=torch.long)

    deg = degree(edge_index[1], x_anon.size(0), dtype=torch.float)
    deg = torch.log(deg + 1).view(-1, 1)

    if feature_mode == 'full':
        x = torch.cat([x_anon, deg], dim=1)
    elif feature_mode == 'anonymous':
        x = x_anon
    elif feature_mode == 'degree':
        x = deg
    else:
        raise ValueError(f'Unknown feature mode: {feature_mode}')

    y = torch.tensor([meta['cc_label_map'][cc_id]], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)


def build_loaders(feature_mode='full'):
    train_data = [build_pyg_graph(cc_id, feature_mode) for cc_id in meta['train_ids']]
    val_data = [build_pyg_graph(cc_id, feature_mode) for cc_id in meta['val_ids']]
    test_data = [build_pyg_graph(cc_id, feature_mode) for cc_id in meta['test_ids']]

    train_pos = [d for d in train_data if d.y.item() == 1]
    train_neg = [d for d in train_data if d.y.item() == 0]
    oversample_factor = max(1, len(train_neg) // len(train_pos))
    train_balanced = train_neg + train_pos * oversample_factor

    train_loader = DataLoader(train_balanced, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

    input_dim = train_data[0].x.size(1)
    return train_loader, val_loader, test_loader, input_dim


for mode in ['full', 'anonymous', 'degree']:
    sample = build_pyg_graph(meta['train_ids'][0], mode)
    print(f"  {mode:>10}: dim = {sample.x.size(1)}")

        full: dim = 44
   anonymous: dim = 43
      degree: dim = 1


## 4. Model (configurable depth + pooling)

In [39]:
class AblationGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, n_layers=3,
                 pool='mean', dropout=DROPOUT):
        super().__init__()
        self.pool_fn = POOL_FN[pool]
        self.dropout = dropout

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()

        self.convs.append(GCNConv(in_channels, hidden))
        self.bns.append(torch.nn.BatchNorm1d(hidden))
        for _ in range(n_layers - 1):
            self.convs.append(GCNConv(hidden, hidden))
            self.bns.append(torch.nn.BatchNorm1d(hidden))

        self.classifier = torch.nn.Linear(hidden, 2)

    def forward(self, x, edge_index, batch):
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            x = bn(conv(x, edge_index))
            if i < len(self.convs) - 1:
                x = F.relu(x)
        x = self.pool_fn(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)

## 5. Training and evaluation

In [40]:
def evaluate(model, loader):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            prob = F.softmax(out, dim=1)[:, 1]
            y_true.extend(batch.y.cpu().numpy())
            y_prob.extend(prob.cpu().numpy())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    has_both = len(np.unique(y_true)) > 1
    return {
        'auroc': roc_auc_score(y_true, y_prob) if has_both else 0.0,
        'f1': f1_score(y_true, (y_prob >= 0.5).astype(int), zero_division=0),
        'y_true': y_true,
        'y_prob': y_prob,
    }


def tune_threshold(y_true, y_prob):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 1.0, 0.05):
        f = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1


def run_ablation(name, feature_mode='full', pool='mean', n_layers=3):
    torch.manual_seed(SEED)

    train_loader, val_loader, test_loader, input_dim = build_loaders(feature_mode)

    model = AblationGCN(
        input_dim, hidden=HIDDEN_DIM, n_layers=n_layers, pool=pool
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE
    )

    best_val_auroc = 0
    best_state = None
    no_improve = 0
    history = []

    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(out, batch.y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        val_m = evaluate(model, val_loader)
        scheduler.step(val_m['auroc'])

        if val_m['auroc'] > best_val_auroc:
            best_val_auroc = val_m['auroc']
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            marker = ' *'
        else:
            no_improve += 1
            marker = ''

        history.append({
            'epoch': epoch, 'loss': avg_loss,
            'val_auroc': val_m['auroc'], 'val_f1': val_m['f1'],
        })

        if epoch <= 3 or epoch % 10 == 0 or marker:
            tag = name[:6].ljust(6)
            print(f"  {tag} Epoch {epoch:3d} | Loss: {avg_loss:.4f} | "
                  f"Val AUROC: {val_m['auroc']:.4f} | Val F1: {val_m['f1']:.4f}{marker}")

        if no_improve >= PATIENCE:
            print(f"  {tag} Early stopping at epoch {epoch}")
            break

    t_elapsed = time.time() - t0
    model.load_state_dict(best_state)

    val_r = evaluate(model, val_loader)
    best_thresh, val_f1_tuned = tune_threshold(val_r['y_true'], val_r['y_prob'])

    test_r = evaluate(model, test_loader)
    y_pred = (test_r['y_prob'] >= best_thresh).astype(int)
    cm = confusion_matrix(test_r['y_true'], y_pred)

    result = {
        'name': name,
        'feature_mode': feature_mode,
        'pool': pool,
        'n_layers': n_layers,
        'input_dim': input_dim,
        'n_params': sum(p.numel() for p in model.parameters()),
        'best_val_auroc': round(best_val_auroc, 4),
        'threshold': best_thresh,
        'test_auroc': round(test_r['auroc'], 4),
        'test_auc_pr': round(average_precision_score(test_r['y_true'], test_r['y_prob']), 4),
        'test_f1': round(f1_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'test_precision': round(precision_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'test_recall': round(recall_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'confusion_matrix': {'tn': int(cm[0,0]), 'fp': int(cm[0,1]),
                             'fn': int(cm[1,0]), 'tp': int(cm[1,1])},
        'epochs_run': len(history),
        'training_time_s': round(t_elapsed, 1),
    }

    print(f"  {name} => Test AUROC: {result['test_auroc']:.4f} | "
          f"F1: {result['test_f1']:.4f} @ thresh={best_thresh:.2f} | "
          f"AUC-PR: {result['test_auc_pr']:.4f} | "
          f"Time: {t_elapsed:.1f}s\n")

    return result

## 6. Ablation A — Feature sets

In [41]:
all_results = {}

print("=" * 70)
print("Ablation A1: Full features (43 anon + 1 degree = 44-dim) [CONTROL]")
print("=" * 70)
all_results['a1_full'] = run_ablation('A1-Full', feature_mode='full')

print("=" * 70)
print("Ablation A2: Anonymous features only (43-dim, no degree)")
print("=" * 70)
all_results['a2_anon'] = run_ablation('A2-Anon', feature_mode='anonymous')

print("=" * 70)
print("Ablation A3: Degree feature only (1-dim, topology only)")
print("=" * 70)
all_results['a3_degree'] = run_ablation('A3-Deg', feature_mode='degree')

Ablation A1: Full features (43 anon + 1 degree = 44-dim) [CONTROL]
  A1-Ful Epoch   1 | Loss: 0.5386 | Val AUROC: 0.8756 | Val F1: 0.3734 *
  A1-Ful Epoch   2 | Loss: 0.4416 | Val AUROC: 0.8608 | Val F1: 0.3849
  A1-Ful Epoch   3 | Loss: 0.3757 | Val AUROC: 0.8534 | Val F1: 0.3654
  A1-Ful Epoch  10 | Loss: 0.1832 | Val AUROC: 0.8392 | Val F1: 0.3830
  A1-Ful Early stopping at epoch 11
  A1-Full => Test AUROC: 0.8525 | F1: 0.4105 @ thresh=0.75 | AUC-PR: 0.4337 | Time: 11.3s

Ablation A2: Anonymous features only (43-dim, no degree)
  A2-Ano Epoch   1 | Loss: 0.5517 | Val AUROC: 0.8624 | Val F1: 0.3517 *
  A2-Ano Epoch   2 | Loss: 0.4455 | Val AUROC: 0.8667 | Val F1: 0.3765 *
  A2-Ano Epoch   3 | Loss: 0.3878 | Val AUROC: 0.8524 | Val F1: 0.3596
  A2-Ano Epoch  10 | Loss: 0.1663 | Val AUROC: 0.8537 | Val F1: 0.4455
  A2-Ano Early stopping at epoch 12
  A2-Anon => Test AUROC: 0.8826 | F1: 0.4790 @ thresh=0.85 | AUC-PR: 0.5481 | Time: 12.1s

Ablation A3: Degree feature only (1-dim, topolog

## 7. Ablation B — Pooling strategy

In [42]:
print("=" * 70)
print("Ablation B1: Global Mean Pool [CONTROL]")
print("=" * 70)
print("  (same as A1-Full, copying result)")
all_results['b1_mean'] = all_results['a1_full']
print()

print("=" * 70)
print("Ablation B2: Global Max Pool")
print("=" * 70)
all_results['b2_max'] = run_ablation('B2-Max', pool='max')

print("=" * 70)
print("Ablation B3: Global Add Pool")
print("=" * 70)
all_results['b3_add'] = run_ablation('B3-Add', pool='add')

Ablation B1: Global Mean Pool [CONTROL]
  (same as A1-Full, copying result)

Ablation B2: Global Max Pool


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


  B2-Max Epoch   1 | Loss: 0.5257 | Val AUROC: 0.8856 | Val F1: 0.4013 *


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


  B2-Max Epoch   2 | Loss: 0.4140 | Val AUROC: 0.8737 | Val F1: 0.3676


...

  B2-Max Epoch   3 | Loss: 0.3462 | Val AUROC: 0.8742 | Val F1: 0.4015


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

  B2-Max Epoch  10 | Loss: 0.1223 | Val AUROC: 0.8486 | Val F1: 0.4286


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


  B2-Max Early stopping at epoch 11
  B2-Max => Test AUROC: 0.8794 | F1: 0.4818 @ thresh=0.75 | AUC-PR: 0.5073 | Time: 11.1s

Ablation B3: Global Add Pool
  B3-Add Epoch   1 | Loss: 0.8781 | Val AUROC: 0.8436 | Val F1: 0.3611 *
  B3-Add Epoch   2 | Loss: 0.5405 | Val AUROC: 0.8647 | Val F1: 0.3678 *
  B3-Add Epoch   3 | Loss: 0.4594 | Val AUROC: 0.8897 | Val F1: 0.3837 *
  B3-Add Epoch  10 | Loss: 0.2199 | Val AUROC: 0.8559 | Val F1: 0.4057
  B3-Add Early stopping at epoch 13
  B3-Add => Test AUROC: 0.8625 | F1: 0.3905 @ thresh=0.80 | AUC-PR: 0.4587 | Time: 12.6s



## 8. Ablation C — Architecture depth

In [43]:
print("=" * 70)
print("Ablation C1: 2 GCN layers")
print("=" * 70)
all_results['c1_2layer'] = run_ablation('C1-2L', n_layers=2)

print("=" * 70)
print("Ablation C2: 3 GCN layers [CONTROL]")
print("=" * 70)
print("  (same as A1-Full, copying result)")
all_results['c2_3layer'] = all_results['a1_full']
print()

print("=" * 70)
print("Ablation C3: 4 GCN layers")
print("=" * 70)
all_results['c3_4layer'] = run_ablation('C3-4L', n_layers=4)

Ablation C1: 2 GCN layers
  C1-2L  Epoch   1 | Loss: 0.5634 | Val AUROC: 0.8645 | Val F1: 0.3724 *
  C1-2L  Epoch   2 | Loss: 0.4845 | Val AUROC: 0.8632 | Val F1: 0.3934
  C1-2L  Epoch   3 | Loss: 0.4397 | Val AUROC: 0.8700 | Val F1: 0.3822 *
  C1-2L  Epoch  10 | Loss: 0.2864 | Val AUROC: 0.8268 | Val F1: 0.3643
  C1-2L  Epoch  13 | Loss: 0.2814 | Val AUROC: 0.8705 | Val F1: 0.4464 *
  C1-2L  Epoch  20 | Loss: 0.2002 | Val AUROC: 0.8486 | Val F1: 0.4444
  C1-2L  Early stopping at epoch 23
  C1-2L => Test AUROC: 0.8791 | F1: 0.5025 @ thresh=0.70 | AUC-PR: 0.5156 | Time: 19.7s

Ablation C2: 3 GCN layers [CONTROL]
  (same as A1-Full, copying result)

Ablation C3: 4 GCN layers
  C3-4L  Epoch   1 | Loss: 0.5410 | Val AUROC: 0.8697 | Val F1: 0.4164 *
  C3-4L  Epoch   2 | Loss: 0.4108 | Val AUROC: 0.8422 | Val F1: 0.3836
  C3-4L  Epoch   3 | Loss: 0.3710 | Val AUROC: 0.8474 | Val F1: 0.3713
  C3-4L  Epoch  10 | Loss: 0.1487 | Val AUROC: 0.8260 | Val F1: 0.4000
  C3-4L  Early stopping at epoch

## 9. Summary table

In [44]:
with open(os.path.join(PROCESSED_PATH, 'training_results.json'), 'r') as f:
    graphsentry = json.load(f)

with open(os.path.join(PROCESSED_PATH, 'baseline_results.json'), 'r') as f:
    baselines = json.load(f)

print("=" * 90)
print("ABLATION RESULTS")
print("=" * 90)
print(f"{'Experiment':<30s} {'Dim':>4s} {'AUROC':>8s} {'AUC-PR':>8s} "
      f"{'F1':>8s} {'Prec':>8s} {'Recall':>8s} {'Thresh':>7s}")
print("-" * 90)

display_order = [
    'a1_full', 'a2_anon', 'a3_degree',
    'b2_max', 'b3_add',
    'c1_2layer', 'c3_4layer',
]

for key in display_order:
    r = all_results[key]
    print(f"{r['name']:<30s} {r['input_dim']:>4d} {r['test_auroc']:>8.4f} {r['test_auc_pr']:>8.4f} "
          f"{r['test_f1']:>8.4f} {r['test_precision']:>8.4f} {r['test_recall']:>8.4f} "
          f"{r['threshold']:>7.2f}")

print("-" * 90)
print()

# Cross-notebook reference
gs = graphsentry['test_metrics']
gcn_bl = baselines['gcn_dataloader']

print("CROSS-NOTEBOOK: Sampling ablation (NB02 vs NB03)")
print("-" * 60)
print(f"  GraphSentry (GraphSAINT GCN)  AUROC: {gs['auroc']:.4f}")
print(f"  Baseline    (DataLoader GCN)  AUROC: {gcn_bl['test_auroc']:.4f}")
print(f"  Delta:                        {gs['auroc'] - gcn_bl['test_auroc']:>+.4f}")

ABLATION RESULTS
Experiment                      Dim    AUROC   AUC-PR       F1     Prec   Recall  Thresh
------------------------------------------------------------------------------------------
A1-Full                          44   0.8525   0.4337   0.4105   0.3052   0.6267    0.75
A2-Anon                          43   0.8826   0.5481   0.4790   0.4348   0.5333    0.85
A3-Deg                            1   0.5038   0.0962   0.1534   0.0857   0.7333    0.50
B2-Max                           44   0.8794   0.5073   0.4818   0.3655   0.7067    0.75
B3-Add                           44   0.8625   0.4587   0.3905   0.3037   0.5467    0.80
C1-2L                            44   0.8791   0.5156   0.5025   0.4032   0.6667    0.70
C3-4L                            44   0.8760   0.4482   0.4484   0.3378   0.6667    0.65
------------------------------------------------------------------------------------------

CROSS-NOTEBOOK: Sampling ablation (NB02 vs NB03)
---------------------------------------

## 10. Delta analysis

In [45]:
control = all_results['a1_full']

print("COMPONENT CONTRIBUTION")
print("=" * 70)
print(f"Control (A1-Full): AUROC={control['test_auroc']:.4f}, F1={control['test_f1']:.4f}")
print()
print(f"{'Ablation':<30s} {'dAUROC':>8s} {'dF1':>8s}")
print("-" * 50)

comparisons = [
    ('a2_anon',   'Remove degree feature'),
    ('a3_degree', 'Remove anonymous features'),
    ('b2_max',    'Mean -> Max pool'),
    ('b3_add',    'Mean -> Add pool'),
    ('c1_2layer', '3 -> 2 layers'),
    ('c3_4layer', '3 -> 4 layers'),
]

for key, desc in comparisons:
    r = all_results[key]
    d_auroc = r['test_auroc'] - control['test_auroc']
    d_f1 = r['test_f1'] - control['test_f1']
    print(f"  {desc:<28s} {d_auroc:>+8.4f} {d_f1:>+8.4f}")

COMPONENT CONTRIBUTION
Control (A1-Full): AUROC=0.8525, F1=0.4105

Ablation                         dAUROC      dF1
--------------------------------------------------
  Remove degree feature         +0.0301  +0.0685
  Remove anonymous features     -0.3487  -0.2571
  Mean -> Max pool              +0.0269  +0.0713
  Mean -> Add pool              +0.0100  -0.0200
  3 -> 2 layers                 +0.0266  +0.0920
  3 -> 4 layers                 +0.0235  +0.0379


## 11. Save results

In [46]:
with open(os.path.join(PROCESSED_PATH, 'ablation_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print("Saved ablation_results.json")
print("\nReady for NB05.")

Saved ablation_results.json

Ready for NB05.
